# CAP182 SS3

In [47]:
import pandas as pd
import numpy as np


## Dataset loading

Purpose
The client supplied multiple datasets containing hospital admissions, patient demographics, clinical information, medication records, outpatient follow-up activity, claims information and procedure data.

The admissions dataset was selected as the primary dataset because it contains the target variable Readmitted_30_Days, indicating whether a patient was readmitted within 30 days of discharge.

The remaining datasets are linked using either Patient_ID or Admission_ID and provide additional predictive information that may improve model performance.

Dataset Overview
Target Variable:

0 = Not Readmitted Within 30 Days
1 = Readmitted Within 30 Days
Dataset size, class distribution and class imbalance are analysed below.

Why CSV?
CSV files are suitable for machine learning workflows because they provide a structured tabular format that can be efficiently loaded into Pandas DataFrames for preprocessing, feature engineering and modelling.

In [48]:
admissions = pd.read_csv("request1_admissions_discharge.csv")
demographics = pd.read_csv("request2_patient_demographics.csv")
clinical = pd.read_csv("request3_medical_history_clinical.csv")
medications = pd.read_csv("request4_medication_treatment.csv")
followup = pd.read_csv("request5_followup_outpatient.csv")
claims = pd.read_csv("request6_claims_billing.csv")
procedures = pd.read_csv("request7_procedures.csv")

print("Admissions:", admissions.shape)
print("Demographics:", demographics.shape)
print("Clinical:", clinical.shape)
print("Medications:", medications.shape)
print("Follow-up:", followup.shape)
print("Claims:", claims.shape)
print("Procedures:", procedures.shape)

print("\nTarget Distribution")
print(admissions["Readmitted_30_Days"].value_counts())

print("\nTarget Percentage")
print(
    admissions["Readmitted_30_Days"]
    .value_counts(normalize=True)
    * 100
)

Admissions: (9174, 13)
Demographics: (5218, 7)
Clinical: (5218, 10)
Medications: (32989, 6)
Follow-up: (4775, 6)
Claims: (9147, 14)
Procedures: (5689, 7)

Target Distribution
Readmitted_30_Days
No     8181
Yes     993
Name: count, dtype: int64

Target Percentage
Readmitted_30_Days
No     89.175932
Yes    10.824068
Name: proportion, dtype: float64


## Dataset processing

Purpose
Data preprocessing was performed to improve data quality and create a single admission-level dataset suitable for machine learning.

The following preprocessing steps were applied:

Duplicate removal
Standardisation of categorical values
Missing value treatment
Outlier handling
Aggregation of one-to-many datasets
Dataset merging
Class imbalance assessment
Train, validation and test splitting
Missing Values
Missing numerical values are replaced using the median because the median is less sensitive to outliers than the mean.

Missing categorical values are replaced using "Unknown".

Outliers
Extreme BMI values were identified and replaced with missing values before imputation because values below 10 or above 80 are medically unrealistic.

Class Imbalance
Class imbalance was assessed because readmission datasets are commonly imbalanced. This influenced the decision to use class weighting during model training.

Aggregation
Medication, follow-up, claims and procedure datasets contain multiple records per admission.

These tables were aggregated to admission level before merging to create a single row per admission suitable for machine learning and to prevent duplicate records.

In [49]:

#___________________REMOVE DUPLICATES______________

demographics = demographics.drop_duplicates()
clinical = clinical.drop_duplicates()


# ______________STANDARDISE TEXT VALUES__________


demographics["Gender"] = (
    demographics["Gender"]
    .astype(str)
    .str.strip()
    .str.title()
)

clinical["Smoking_Status"] = (
    clinical["Smoking_Status"]
    .astype(str)
    .str.strip()
    .str.title()
)


# ___________HANDLE BMI OUTLIERS__________


clinical.loc[
    (clinical["BMI"] < 10) |
    (clinical["BMI"] > 80),
    "BMI"
] = np.nan


#__________ MEDICATION AGGREGATION_________


med_agg = (
    medications
    .groupby("Admission_ID")
    .agg(
        Medication_Count=("Medication_Name", "count"),
        Average_Daily_Dosage=("Daily_Dosage", "mean"),
        Number_of_Medications=("Number_of_Medications", "max"),
        Treatment_Completed=("Treatment_Completed", "max"),
        Follow_Up_Required=("Follow_Up_Required", "max")
    )
    .reset_index()
)


# ___________FOLLOW-UP AGGREGATION_____________


followup_agg = (
    followup
    .groupby("Admission_ID")
    .agg(
        Attended=("Attended", "max"),
        Days_to_Follow_Up=("Days_to_Follow_Up", "mean"),
        Missed_Appointments=("Missed_Appointments", "max")
    )
    .reset_index()
)


# _____________CLAIMS AGGREGATION_____________


claims_agg = (
    claims
    .groupby("Admission_ID")
    .agg(
        Total_Claim_Value=("Total_Claim_Value", "sum"),
        Patient_Copayment=("Patient_Copayment", "sum"),
        Total_Line_Items=("Total_Line_Items", "sum"),
        Settlement_Amount=("Settlement_Amount", "sum"),
        Query_Flag=("Query_Flag", "max")
    )
    .reset_index()
)


# _____________PROCEDURE AGGREGATION__________


procedure_agg = (
    procedures
    .groupby("Admission_ID")
    .agg(
        Procedure_Count=("Procedure_Code", "count"),
        Total_Theatre_Time=("Theatre_Time_Minutes", "sum")
    )
    .reset_index()
)




In [50]:
# ______________MERGE ALL DATASETS___________


df = admissions.merge(
    demographics,
    on="Patient_ID",
    how="left"
)

df = df.merge(
    clinical,
    on="Patient_ID",
    how="left"
)

df = df.merge(
    med_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    followup_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    claims_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    procedure_agg,
    on="Admission_ID",
    how="left"
)



In [51]:

# __________HANDLE MISSING VALUES_____________


for col in df.select_dtypes(include=np.number):
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")



In [52]:

#____________ TARGET ENCODING


df["Readmitted_30_Days"] = (
    df["Readmitted_30_Days"]
    .astype(str)
    .str.strip()
    .map({
        "No": 0,
        "Yes": 1
    })
)

print("\nEncoded Target Variable")
print(df["Readmitted_30_Days"].value_counts())


Encoded Target Variable
Readmitted_30_Days
0    8181
1     993
Name: count, dtype: int64


In [53]:

# _________DATA QUALITY CHECKS_________


print("Final Dataset Shape:", df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nClass Distribution")
print(df["Readmitted_30_Days"].value_counts())

print("\nClass Percentage")
print(
    df["Readmitted_30_Days"]
    .value_counts(normalize=True)
    * 100
)

Final Dataset Shape: (9174, 43)

Missing Values
Patient_ID                      0
Admission_ID                    0
Admission_Date                  0
Discharge_Date                  0
Length_of_Stay                  0
Hospital_ID                     0
Ward_Type                       0
Admission_Type                  0
Procedure_Code                  0
Procedure_Description           0
Discharge_Status                0
Previous_Admissions             0
Readmitted_30_Days              0
Age                             0
Gender                          0
Province                        0
Medical_Scheme                  0
Membership_Type                 0
Employment_Status               0
Chronic_Diabetes                0
Chronic_Hypertension            0
Chronic_Heart_Disease           0
Chronic_Asthma                  0
Number_of_Chronic_Conditions    0
BMI                             0
Smoking_Status                  0
Allergies                       0
Risk_Score                      0


## Model training

Model Selection
Logistic Regression was selected because it was the best-performing model during SS2 and therefore became the final recommended model.

Using the same model ensures consistency between SS2 and SS3.

Hyperparameters
max_iter = 2000

A larger number of iterations ensures convergence during optimisation.

class_weight = "balanced"

This compensates for the class imbalance identified during preprocessing.

random_state = 42

This ensures reproducibility.

Data Splitting
The dataset was divided into:

Training Set = 70%
Validation Set = 15%
Test Set = 15%
This allows model training, tuning and unbiased final evaluation.

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import joblib

In [55]:

y = df["Readmitted_30_Days"]

X = df.drop(
    columns=[
        "Readmitted_30_Days",
        "Admission_ID",
        "Patient_ID"
    ],
    errors="ignore"
)

# 70 / 15 / 15 split

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    stratify=y_temp,
    random_state=42
)

numeric_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]
            ),
            numeric_cols
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            categorical_cols
        )
    ]
)

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

model.fit(
    X_train,
    y_train
)

joblib.dump(
    model,
    "logistic_regression_model.pkl"
)

print("Model Training Complete")

Model Training Complete


## Model validation

Purpose
The final model is evaluated using previously unseen test data.

Metrics used:

Accuracy
Precision
Recall
F1 Score
ROC-AUC
Recall is particularly important because the business objective is to identify patients at risk of readmission before discharge.

Failing to identify a high-risk patient may lead to increased costs, higher readmission rates and poorer patient outcomes.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [56]:


predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)

print("\nClassification Report")
print(classification_report(y_test, predictions))

results = pd.DataFrame({
    "actual": y_test,
    "prediction": predictions,
    "probability": probabilities
})

results.to_csv(
    "SS3_predictions.csv",
    index=False
)

Accuracy : 0.8300653594771242
Precision: 0.2702702702702703
Recall   : 0.33557046979865773
F1 Score : 0.2994011976047904
ROC AUC  : 0.7139289071551932

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.89      0.90      1228
           1       0.27      0.34      0.30       149

    accuracy                           0.83      1377
   macro avg       0.59      0.61      0.60      1377
weighted avg       0.85      0.83      0.84      1377

